In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [6]:
!nvidia-smi

Tue Sep  8 07:29:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
import torch

print("CUDA Available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU: Tesla T4


In [8]:
!pip install -q \
    transformers \
    datasets \
    accelerate \
    peft \
    bitsandbytes \
    trl \
    sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 41.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 43.9 MB/s eta 0:00:00


In [9]:
import torch
import pandas as pd

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

In [10]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [11]:
model_name =  "Qwen/Qwen3-4B"

In [12]:
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [13]:
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [14]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [15]:
from datasets import load_dataset

dataset = load_dataset(
    "euclaise/writingprompts",
    split="train"
)

dataset

README.md:   0%|          | 0.00/837 [00:00<?, ?B/s]

data/train-00000-of-00002-105e07cb0d1994(…):   0%|          | 0.00/272M [00:00<?, ?B/s]

data/train-00001-of-00002-4fdb982c110564(…):   0%|          | 0.00/272M [00:00<?, ?B/s]

data/test-00000-of-00001-16503b0c26ed00c(…):   0%|          | 0.00/30.0M [00:00<?, ?B/s]

data/validation-00000-of-00001-137b93e1e(…):   0%|          | 0.00/30.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/272600 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/15138 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/15620 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'story'],
    num_rows: 272600
})

In [16]:
dataset = dataset.shuffle(seed=42).select(range(5000))
def format_example(example):
    
    text = f"""### Instruction:
Transform the following context into an engaging and coherent narrative story.

### Context:
{example['prompt']}

### Story:
{example['story']}"""
    
    return {"text": text}

In [17]:
formatted_dataset = dataset.map(
    format_example
)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [18]:
dataset_split = formatted_dataset.train_test_split(
    test_size=0.1,
    seed=42
)

train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

print("Train:", len(train_dataset))
print("Validation:", len(eval_dataset))

Train: 4500
Validation: 500


In [19]:
from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model
)


In [20]:
model = prepare_model_for_kbit_training(model)

In [21]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [22]:
model = prepare_model_for_kbit_training(model)

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


In [23]:
MAX_LENGTH = 512

def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )

In [24]:
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_eval = eval_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=eval_dataset.column_names
)

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [25]:
def add_labels(example):
    example["labels"] = example["input_ids"].copy()
    return example

In [26]:
tokenized_train = tokenized_train.map(add_labels)
tokenized_eval = tokenized_eval.map(add_labels)

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [27]:
def mask_padding(example):
    labels = [
        -100 if token == tokenizer.pad_token_id else token
        for token in example["labels"]
    ]
    
    example["labels"] = labels
    return example

In [28]:
tokenized_train = tokenized_train.map(mask_padding)
tokenized_eval = tokenized_eval.map(mask_padding)

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [29]:
from transformers import TrainingArguments
from transformers import Trainer

In [30]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/qwen3-story-lora",

    num_train_epochs=2,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    fp16=True,

    logging_steps=10,

    # =========================
    # AUTO SAVE CHECKPOINTS
    # =========================
    save_strategy="steps",
    save_steps=25,          # يحفظ كل 25 خطوة
    save_total_limit=3,     # يحتفظ بآخر 3 checkpoints

    # مهم جدًا لو التدريب وقف
    save_only_model=False,

    report_to="none"
)

In [31]:
test_prompt = """### Instruction:
Transform the following context into an engaging and coherent narrative story.

### Context:
A new artificial intelligence system was introduced today. 
Researchers said it could improve education and healthcare.

### Story:
"""

In [32]:
inputs = tokenizer(
    test_prompt,
    return_tensors="pt"
).to(model.device)

base_output = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.8,
    top_p=0.9
)

print(tokenizer.decode(base_output[0], skip_special_tokens=True))

### Instruction:
Transform the following context into an engaging and coherent narrative story.

### Context:
A new artificial intelligence system was introduced today. 
Researchers said it could improve education and healthcare.

### Story:
Once upon a time, there was a smart AI named Alex. It was the latest invention that could help people learn and heal. In the town of Techville, a group of scientists and teachers worked together to create Alex. They wanted to use it to make education better and healthcare more efficient. Alex was a bit shy at first, but with time, it became more confident and helpful. It helped students understand difficult subjects by giving them personalized lessons. In the hospital, it assisted doctors in diagnosing diseases more accurately. People in Techville were amazed by Alex’s abilities. The town became famous for its innovative AI. 

### Instruction:
Transform the following context into an engaging and coherent narrative story.

### Context:
A new artific

In [33]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=data_collator
)

In [34]:
train_result = trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such a

Step,Training Loss
10,3.123270
20,2.830924
30,2.733958
40,2.708123
50,2.656211
60,2.741291
70,2.685836
80,2.726096
90,2.694715
100,2.738354


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

In [38]:
import os

print(os.listdir("/kaggle/working/qwen3-story-lora"))

['checkpoint-1100', 'checkpoint-1125', 'checkpoint-1126']


In [39]:
# Save final model

SAVE_PATH = "/kaggle/working/qwen3-final-adapter"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Final model saved!")

Final model saved!


In [40]:
import shutil

shutil.make_archive(
    "/kaggle/working/qwen3-final-adapter",
    "zip",
    SAVE_PATH
)

print("ZIP created successfully!")

ZIP created successfully!


In [41]:
import os

SAVE_PATH = "/kaggle/working/qwen3-final-adapter"

print(os.listdir(SAVE_PATH))

['adapter_config.json', 'README.md', 'tokenizer_config.json', 'adapter_model.safetensors', 'training_args.bin', 'tokenizer.json', 'chat_template.jinja']


In [42]:
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2560)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2560, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [ ]:
def generate_story(prompt, max_new_tokens=500):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=1000,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        repetition_penalty=1.1,
        no_repeat_ngram_size=3
)

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response

In [55]:
planning_prompt = """
Create a short story outline using exactly these three events.

Event 1:
A football club fires its MANAGER after only three matchdays.

Event 2:
Artificial intelligence is increasingly important in EDUCATION.

Event 3:
SCIENTISTS announce a new discovery related to CLIMATE CHANGE.

Create exactly 5 story points.

Requirements:
- Point 1 must include the football manager being fired.
- Point 2 and 3 must involve AI in education.
- Point 4 must involve scientists and a climate change discovery.
- Point 5 must connect all storylines.

Return only the outline.
"""

In [56]:
outline = generate_story(
    planning_prompt,
    max_new_tokens=400
)

print(outline)

 **Title: The Great Reset** 
 
 **Point 1:** A small English town has just lost its football team to a massive American corporation. They had been losing for years, and their fans had become disillusioned with the team's lack of progress. When the manager was fired, the townspeople were devastated, thinking that it would be the end of football in their community. But when the president announced his plans to start an AI-driven sports league, everyone began to hope again. 
 
 **Points 2 & 3:** With the start of this new sports league looming, students across the country began to turn to artificial intelligence for help learning how to play football. They trained themselves against AI opponents, and soon they were dominating the leagues. Meanwhile, scientists continued to work on their research into climate change. 
 
 
 **Point4:** In a shocking development, scientists announced that the ice sheets around Antarctica are melting faster than expected. This could mean that global warming w

In [57]:
story_prompt = f"""
Write a coherent fictional story based strictly on this outline:

{outline}

Requirements:

- The main character must be a football MANAGER, not a player.
- He must be fired after exactly three matchdays.
- Artificial intelligence must be explicitly used in education.
- Scientists must explicitly announce a climate change discovery.
- Connect all events naturally.
- Use consistent character names.
- Do not invent unrelated major events.
- Write approximately 700 words.
- Write only the story.
"""

In [58]:
story = generate_story(
    story_prompt,
    max_new_tokens=900
)

print(story)

 *I am sorry if you've already read my answer to your query before - I'm trying to write answers fast and I tend to get confused by other questions* 
 
 It's the year 2018. Football is back in fashion, thanks to a new government initiative called the Great Reset. The government is trying to boost morale amongst the general populace by creating a new football league, funded entirely by tax money. It's not going to last long, though. 
 
 `` We're running out of time,'' said the President as he sat down at the table with his most trusted advisors. `` We need to move quickly.'' 
 
 The President turned towards the head of the state sports commission. `` You have a plan?'' 
 
 `` Yes, Mr. President. I'll present one in ten minutes.'' 
 
 *The head of state sports committee walks into the room, followed by four other men. All of them wear black suits, except for the man who stands behind the Head Sports Commissioner. He wears a white suit with red trim.* 
 
 `` Mr. Presi... uh... President.'

In [59]:
del model

In [60]:
import gc
import torch

del model

gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared successfully!")

NameError: name 'model' is not defined

In [61]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model.eval()

print("Base Qwen3 loaded successfully!")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Base Qwen3 loaded successfully!


In [62]:
def generate_story(prompt, max_new_tokens=1000):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            no_repeat_ngram_size=3
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response

In [63]:
prompt = """
You are a professional fiction writer.

Write ONE complete fictional story based on ALL THREE events below.

IMPORTANT:
All three events have EQUAL importance.
Do not spend most of the story on only one event.

The story must explicitly include:

EVENT 1:
A football club fires its MANAGER after only three matchdays.

EVENT 2:
Artificial intelligence is becoming increasingly important in EDUCATION.
The story must specifically involve AI being used in education or learning.

EVENT 3:
Scientists announce a new discovery related to CLIMATE CHANGE.
The story must specifically mention scientists and their climate-related discovery.

STORY STRUCTURE:

Beginning:
Introduce the football manager and his dismissal after three matchdays.

Middle:
Connect the characters to artificial intelligence being used in education.

Ending:
Connect the story to scientists announcing a new discovery related to climate change.

Connect all events naturally through the same characters or storyline.

Do not omit any event.
Do not replace an event with a similar concept.

Write ONLY the complete fictional story.
"""

In [64]:
base_story = generate_story(
    prompt,
    max_new_tokens=1000
)

print(base_story)

**Title: "The Algorithm and the Ball"**

In the heart of a bustling city, where the hum of traffic blended with the distant roar of fans, stood the St. Alaric Football Club. Known for its fierce rivalries and passionate supporters, the team had been a staple of local sports culture for decades. At the helm was Marcus Voss, a charismatic but ambitious manager who had just been appointed to lead the club after a promising career as a player and coach. His vision was clear: to bring the team back to glory, to make history once again.

Three matchdays into the season, however, things took a turn. The team’s performance was dismally poor—three straight losses, a humiliating 4-0 defeat against a lower-tier side that had barely made the league. Fans were furious, media outlets were questioning his tactics, and the board of directors met in a tense room, their faces etched with disappointment. After hours of deliberation, the decision was made: Marcus V oss was fired. The club needed a fresh s

In [65]:
def build_story_prompt(events):
    
    events_text = ""
    
    for i, event in enumerate(events, 1):
        
        title = event.get("title", "")
        content = event.get("content", "")
        source = event.get("source", "")
        
        events_text += f"""
EVENT {i}

Title:
{title}

Description:
{content}

Source:
{source}

------------------------
"""
    
    prompt = f"""
You are a professional fiction writer.

Your task is to transform the following real-world trending events into ONE
creative, engaging, and coherent fictional story.

IMPORTANT RULES:

1. Include ALL events provided.
2. Do not omit important events.
3. Preserve the main meaning of every event.
4. Connect all events naturally through characters and plot.
5. Do not introduce unrelated major events.
6. Create consistent characters and locations.
7. Do not change the meaning of the original events.
8. The final output must be a fictional narrative story.
9. Write ONLY the story.
10. Do not mention that you were given events or instructions.

TRENDING EVENTS:

{events_text}

Now write one coherent fictional story that naturally connects all events.
"""
    
    return prompt

In [66]:
def generate_story_from_events(
    events,
    max_new_tokens=1000
):
    
    prompt = build_story_prompt(events)
    
    story = generate_story(
        prompt,
        max_new_tokens=max_new_tokens
    )
    
    return story

In [67]:
events = [
    
    {
        "id": "news_001",
        "source": "Onefootball",
        "title": "Football club fires manager after three matchdays",
        "content": "A football club has unexpectedly fired its manager after only three matchdays.",
        "date": "2026-09-06"
    },
    
    {
        "id": "news_002",
        "source": "Tech News",
        "title": "Artificial intelligence becomes important in education",
        "content": "Artificial intelligence is becoming increasingly important in education and personalized learning.",
        "date": "2026-09-06"
    },
    
    {
        "id": "news_003",
        "source": "Science News",
        "title": "Scientists announce climate change discovery",
        "content": "Scientists announced new discoveries related to climate change.",
        "date": "2026-09-06"
    }
]

In [68]:
final_story = generate_story_from_events(events)

print(final_story)

In the bustling city of Nova Haven, where the skyline was a jagged mix of steel and glass, the world turned on a dime. At the heart of the city stood the iconic Nova FC stadium, a place where dreams were made and legends were born. But on a rainy Wednesday morning, the very foundation of that dream trembled as the club’s manager, Marcus Voss, was abruptly fired after just three match days. The decision sent shockwaves through the sports community, with fans and analysts alike speculating about the reasons behind the sudden move.

Marcus had been a charismatic figure, known for his fiery speeches and unshakable belief in his team. Yet, within weeks, his leadership had come under scrutiny. The players, once eager and unified, began to drift apart. The coaching staff whispered of inconsistencies in strategy, and the fans grew restless. It was as if the team had lost its heartbeat, and Marcus, the man who was supposed to keep it beating, had failed to do so.

Meanwhile, across the city, in

In [72]:
def validate_story_events(events, story):

    story_lower = story.lower()

    validation_results = []

    event_keywords = {

        "football": [
            "football",
            "club",
            "manager",
            "coach",
            "fired",
            "sacked",
            "dismissed",
            "three match"
        ],

        "education": [
            "artificial intelligence",
            "ai",
            "education",
            "school",
            "student",
            "learning",
            "teacher",
            "personalized learning"
        ],

        "climate": [
            "scientist",
            "climate",
            "climate change",
            "discovery",
            "research",
            "global warming",
            "environment",
            "carbon"
        ]
    }

    for i, event in enumerate(events, 1):

        text = (
            event.get("title", "") + " " +
            event.get("content", "")
        ).lower()

        # Detect event category
        if any(word in text for word in ["football", "manager", "matchday", "club"]):
            category = "football"

        elif any(word in text for word in ["artificial intelligence", "ai", "education", "learning"]):
            category = "education"

        elif any(word in text for word in ["climate", "scientist", "environment"]):
            category = "climate"

        else:
            category = None

        keywords = event_keywords.get(category, [])

        matched_keywords = [
            keyword
            for keyword in keywords
            if keyword in story_lower
        ]

        coverage_score = (
            len(matched_keywords) / len(keywords) * 100
            if keywords else 0
        )

        validation_results.append({
            "event_id": event.get("id", f"event_{i}"),
            "title": event.get("title", ""),
            "category": category,
            "matched_keywords": matched_keywords,
            "coverage_score": round(coverage_score, 2)
        })

    return validation_results

In [73]:
validation_results = validate_story_events(
    events,
    final_story
)

for result in validation_results:
    
    print("=" * 60)
    print("Event:", result["title"])
    print("Matched Keywords:", result["matched_keywords"])
    print("Coverage Score:", result["coverage_score"], "%")

Event: Football club fires manager after three matchdays
Matched Keywords: ['club', 'manager', 'coach', 'fired', 'three match']
Coverage Score: 62.5 %
Event: Artificial intelligence becomes important in education
Matched Keywords: ['ai', 'education', 'school', 'student', 'learning', 'teacher']
Coverage Score: 75.0 %
Event: Scientists announce climate change discovery
Matched Keywords: ['scientist', 'climate', 'climate change', 'discovery']
Coverage Score: 50.0 %


In [74]:
overall_score = sum(
    result["coverage_score"]
    for result in validation_results
) / len(validation_results)

print("=" * 60)
print("OVERALL EVENT COVERAGE SCORE:", round(overall_score, 2), "%")

OVERALL EVENT COVERAGE SCORE: 62.5 %


In [75]:
!pip install -q sentence-transformers

In [76]:
from sentence_transformers import SentenceTransformer, util
import re
import torch

In [77]:
semantic_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Semantic model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Semantic model loaded successfully!


In [78]:
def semantic_validate_events(events, story):
    
    # Split story into sentences
    sentences = re.split(
        r'(?<=[.!?])\s+',
        story
    )
    
    # Remove very short sentences
    sentences = [
        sentence.strip()
        for sentence in sentences
        if len(sentence.strip()) > 20
    ]
    
    # Encode all story sentences once
    story_embeddings = semantic_model.encode(
        sentences,
        convert_to_tensor=True
    )
    
    results = []
    
    for event in events:
        
        # Combine title + content
        event_text = (
            event.get("title", "") +
            ". " +
            event.get("content", "")
        )
        
        # Encode event
        event_embedding = semantic_model.encode(
            event_text,
            convert_to_tensor=True
        )
        
        # Calculate similarity with every sentence
        similarities = util.cos_sim(
            event_embedding,
            story_embeddings
        )[0]
        
        # Get best matching sentence
        best_index = torch.argmax(similarities).item()
        
        best_score = similarities[
            best_index
        ].item()
        
        best_sentence = sentences[
            best_index
        ]
        
        results.append({
            "event_id": event.get("id"),
            "event": event.get("title"),
            "best_matching_sentence": best_sentence,
            "semantic_similarity": round(
                best_score * 100,
                2
            )
        })
    
    return results

In [79]:
semantic_results = semantic_validate_events(
    events,
    final_story
)

for result in semantic_results:
    
    print("=" * 70)
    print("EVENT:")
    print(result["event"])
    
    print("\nBEST MATCHING STORY SENTENCE:")
    print(result["best_matching_sentence"])
    
    print(
        "\nSEMANTIC SIMILARITY:",
        result["semantic_similarity"],
        "%"
    )

EVENT:
Football club fires manager after three matchdays

BEST MATCHING STORY SENTENCE:
But on a rainy Wednesday morning, the very foundation of that dream trembled as the club’s manager, Marcus Voss, was abruptly fired after just three match days.

SEMANTIC SIMILARITY: 52.42 %
EVENT:
Artificial intelligence becomes important in education

BEST MATCHING STORY SENTENCE:
Schools were beginning to integrate the technology into their curricula, and teachers found themselves marveling at how much more effective lessons could be when tailored to individual needs.

SEMANTIC SIMILARITY: 51.82 %
EVENT:
Scientists announce climate change discovery

BEST MATCHING STORY SENTENCE:
Meanwhile back in the tech world, scientists at the Global Climate Institute had made a groundbreaking discovery.

SEMANTIC SIMILARITY: 67.11 %


In [80]:
overall_semantic_score = sum(
    result["semantic_similarity"]
    for result in semantic_results
) / len(semantic_results)

print("=" * 70)
print(
    "OVERALL SEMANTIC COVERAGE SCORE:",
    round(overall_semantic_score, 2),
    "%"
)

OVERALL SEMANTIC COVERAGE SCORE: 57.12 %


In [81]:
def classify_coverage(score):
    
    if score >= 0.50:
        return "Covered"
    
    elif score >= 0.35:
        return "Partially Covered"
    
    else:
        return "Not Covered"

In [82]:
for result in semantic_results:
    
    score = result["semantic_similarity"] / 100
    
    result["coverage_status"] = classify_coverage(score)
    
    print("=" * 70)
    print("EVENT:", result["event"])
    print("SIMILARITY:", result["semantic_similarity"], "%")
    print("STATUS:", result["coverage_status"])

EVENT: Football club fires manager after three matchdays
SIMILARITY: 52.42 %
STATUS: Covered
EVENT: Artificial intelligence becomes important in education
SIMILARITY: 51.82 %
STATUS: Covered
EVENT: Scientists announce climate change discovery
SIMILARITY: 67.11 %
STATUS: Covered


In [83]:
overall_semantic_score = sum(
    result["semantic_similarity"]
    for result in semantic_results
) / len(semantic_results)

print(
    f"Overall Semantic Coverage Score: "
    f"{overall_semantic_score:.2f}%"
)

Overall Semantic Coverage Score: 57.12%


In [84]:
!pip install -q gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
typer 0.24.2 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


In [85]:
from gtts import gTTS
from IPython.display import Audio, display

In [86]:
tts = gTTS(
    text=final_story,
    lang="en",
    slow=False
)

audio_path = "/kaggle/working/generated_story.mp3"

tts.save(audio_path)

print("Audio generated successfully!")

Audio generated successfully!


In [87]:
display(Audio(audio_path))

In [89]:
import os

print(os.path.exists("/kaggle/working/generated_story.mp3"))
print(os.path.getsize("/kaggle/working/generated_story.mp3"))

True
2649024


In [90]:
from IPython.display import display, HTML

display(HTML("""
<a href="/kaggle/working/generated_story.mp3" download>
    Download Generated Story Audio
</a>
"""))

In [91]:
import shutil

shutil.make_archive(
    "/kaggle/working/generated_story",
    "zip",
    "/kaggle/working",
    "generated_story.mp3"
)

print("ZIP created successfully!")

ZIP created successfully!


In [92]:
from IPython.display import FileLink

FileLink("/kaggle/working/generated_story.zip")

/kaggle/working/generated_story.zip

In [93]:
import os

print(os.listdir("/kaggle/working"))

['.virtual_documents', 'qwen3-story-lora', 'qwen3-final-adapter', 'generated_story.mp3', 'generated_story.zip', 'qwen3-final-adapter.zip']


In [94]:
import shutil

# اعمل فولدر output
os.makedirs("/kaggle/working/output", exist_ok=True)

# انسخ ملف الصوت
shutil.copy(
    "/kaggle/working/generated_story.mp3",
    "/kaggle/working/output/generated_story.mp3"
)

print(os.listdir("/kaggle/working/output"))

['generated_story.mp3']
